# PORT Account FI SecMaster — Job Overview (Aug–Nov 2025)

**Date:** 2025-11-29  
**Purpose:** Which PORT jobs drive FI SecMaster billed volume, compared across specific snapshot dates (Aug–Nov 2025).  
**Version:** analysis_v1 @ 2025-11

In [0]:
%run "../report_core"

In [0]:
# Investigation: which PORT jobs drive FI SecMaster billed volume, compared across specific snapshot dates (Aug-Nov 2025).
# Dates are fixed to that investigation; edit the list to re-run for another period.
required_data = (
  sd_df
  .where(col("ACCOUNT_ID") == PORT_ACCOUNT)
  .where(col("DATE").isin([
    "2025-08-01",
    "2025-09-01",
    # "2025-10-01",
    # "2025-10-15",
    # "2025-10-16",
    "2025-10-31",
    "2025-11-29"]))
  .where(col("BILLABLE_PRODUCT") == Billable_Products.FI_SECM)
  .groupBy("DATE", "OUTPUT_ID", "CTRB_BILLING").agg(F.countDistinct("ID_BB_UNIQUE").alias("Count"))
  .orderBy("DATE", "CTRB_BILLING")
)

(
  required_data
  .withColumn("env", F.element_at(F.split("output_id", "_"), -1))
  .groupBy("OUTPUT_ID", "CTRB_BILLING", "ENV")
  .pivot("date")
  .agg(F.first("count"))
  .orderBy(
    "CTRB_BILLING", 
    F.when(F.lower(F.col("env")) == "dev", 1).when(F.lower(F.col("env")) == "uat", 2).when(F.lower(F.col("env")) == "prod", 3).otherwise(4), 
    "OUTPUT_ID")
).display()